#Silver Layer

here we will read data from  bronze layer .. do some transformations .. then write it in the Silver Layer

#Reading

In [0]:
df = spark.table("workspace.bronze.cust_info")

In [0]:
df.display()

#Transformation

In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col
#remove leading and trailing spaces from all string columns
for field in df.schema:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))
df.display() 

In [0]:
#Normalization of the data
import pyspark.sql.functions as F

df = (
    df.withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
            .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
            .otherwise("n/a")
        )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "M", "Male")
            .when(F.upper(F.col("cst_gndr")) == "F", "Female")
            .otherwise("n/a")
        )
)

In [0]:
df.display()

In [0]:
#rename all the columns
df = df.withColumnRenamed("cst_id", "customer_id") \
       .withColumnRenamed("cst_key", "customer_key") \
       .withColumnRenamed("cst_firstname", "customer_firstname") \
       .withColumnRenamed("cst_lastname", "customer_lastname") \
       .withColumnRenamed("cst_marital_status", "customer_marital_status") \
       .withColumnRenamed("cst_gndr", "customer_gender") \
       .withColumnRenamed("cst_create_date", "customer_create_date")
df.display()

#Writing

In [0]:
#write the dataframe to the silver table
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.cust_info")

#check

In [0]:
%sql
select * from workspace.silver.cust_info

so the data now is in the silver layer .. and we will do the same with all tables